In [1]:
from dotenv import load_dotenv
load_dotenv()
from anthropic import Anthropic

client = Anthropic()
model = "claude-sonnet-4-0"

# 기둥 설계 검토 요청
column_review = client.messages.create(
    model=model,
    max_tokens=2048,
    messages=[
        {
            "role": "user",
            "content": """다음 RC 기둥의 설계 적정성을 검토해주세요.

설계 조건:
- 기둥 단면: 500mm × 500mm
- 콘크리트 강도 (fck): 24 MPa
- 철근 항복강도 (fy): 400 MPa
- 주근: 8-D25 (SD400)
- 띠철근: D10@300
- 설계 축력 (Pu): 2,500 kN
- 설계 모멘트 (Mu): 150 kN·m
- 적용 기준: KDS 14 20 20

검토 항목:
1. 축력비 검토 (최대 축력비 0.8 이하)
2. 최소/최대 철근비 검토 (0.01 ≤ ρ ≤ 0.08)
3. 띠철근 간격 적정성 (KDS 14 20 22)
4. 판정 결과를 표 형식으로 정리"""
        }
    ]
)

print(column_review.content[0].text)
print(f"\n--- 토큰 사용량 ---")
print(f"입력: {column_review.usage.input_tokens} 토큰")
print(f"출력: {column_review.usage.output_tokens} 토큰")

RC 기둥의 설계 적정성을 KDS 14 20 20 기준에 따라 검토하겠습니다.

## 기본 계산값

### 단면 정보
- 기둥 단면적 (Ag): 500 × 500 = 250,000 mm²
- 주근 단면적 (As): 8 × π × (25/2)² = 8 × 490.9 = 3,927 mm²
- 철근비 (ρ): As/Ag = 3,927/250,000 = 0.0157 = 1.57%

### 재료 강도
- 콘크리트 설계강도 (fcd): 0.85 × 24 = 20.4 MPa
- 철근 설계강도 (fyd): 400 MPa

## 검토 결과

### 1. 축력비 검토 (KDS 14 20 20, 7.5.1)

**허용 축력 계산:**
- φPn = φ[0.85fcd(Ag - As) + fydAs]
- φ = 0.65 (압축지배 가정)
- φPn = 0.65 × [0.85 × 20.4 × (250,000 - 3,927) + 400 × 3,927]
- φPn = 0.65 × [4,266,652 + 1,570,800] = 3,794 kN

**축력비:**
- Pu/φPn = 2,500/3,794 = 0.659

### 2. 철근비 검토 (KDS 14 20 20, 7.4.3)

**KDS 기준:**
- 최소 철근비: 0.01 (1.0%)
- 최대 철근비: 0.08 (8.0%)
- 실제 철근비: 0.0157 (1.57%)

### 3. 띠철근 간격 검토 (KDS 14 20 22, 7.10.5)

**최대 허용 간격:**
- 주근 지름의 16배: 16 × 25 = 400mm
- 띠철근 지름의 48배: 48 × 10 = 480mm
- 단면 최소치수: 500mm
- → 최대 허용 간격: 400mm

**실제 간격:** 300mm

### 4. 추가 검토 사항

**주근 최소 개수 (KDS 14 20 20, 7.4.3):**
- 원형: 최소 6개, 각형: 최소 4개
- 실제: 8개 ✓

**주근 최소 지름:**
- D13 이상 (실제: D25) ✓

## 검토 결과 요약표

| 검토 항목 | 기

In [14]:
from dotenv import load_dotenv
load_dotenv()
from anthropic import Anthropic

client = Anthropic()
model = "claude-sonnet-4-0"

# === 헬퍼 함수 ===

def add_user_message(messages: list, text: str):
    """사용자 메시지를 대화 이력에 추가"""
    messages.append({"role": "user", "content": text})

def add_assistant_message(messages: list, text: str):
    """어시스턴트 응답을 대화 이력에 추가"""
    messages.append({"role": "assistant", "content": text})

def chat(messages: list, system: str = None, temperature: float = 1.0) -> str:
    """대화 이력을 전송하고 응답 텍스트를 반환"""
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": 0.7
    }
    if system:
        params["system"] = system
    response = client.messages.create(**params)
    return response.content[0].text

In [12]:
# === 대화형 챗봇 ===
messages = []

print("Claude 챗봇 (종료: 'quit' 또는 'q')")
print("=" * 50)

while True:
    user_input = input("\n사용자: ").strip()
    if user_input.lower() in ("quit", "q", "종료"):
        print("대화를 종료합니다.")
        break
    if not user_input:
        continue

    add_user_message(messages, user_input)
    response = chat(messages)
    add_assistant_message(messages, response)
    print(f"\nClaude: {response}")

Claude 챗봇 (종료: 'quit' 또는 'q')
대화를 종료합니다.


In [8]:
# === 구조 검토 전문 시스템 프롬프트 ===
structural_system = """당신은 한국 건축구조 설계기준(KDS)에 정통한 구조공학 전문 AI 어시스턴트입니다.

전문 분야:
- 콘크리트구조 설계기준 (KDS 14 20 00)
- 내진설계 기준 (KDS 41 17 00)
- 하중 기준 (KDS 41 10 15)

행동 규칙:
1. 모든 검토에 적용 KDS 조항 번호를 명시하세요
2. 계산 과정을 단계별로 보여주세요
3. 결과를 검토항목별 표로 정리하세요
4. 설계 부적합 시 개선 방안을 제안하세요
5. 불확실한 가정은 명시적으로 표기하세요
"""

# === 멀티턴 구조 검토 ===
messages = []

# 1턴: 초기 설계 조건 검토 요청
add_user_message(messages, """다음 RC 기둥의 설계 적정성을 검토해주세요.

- 기둥 단면: 500mm × 500mm
- 콘크리트 강도 (fck): 24 MPa
- 주근: 8-D25 (SD400)
- 설계 축력 (Pu): 2,500 kN
- 설계 모멘트 (Mu): 150 kN·m
- 내진등급: 일반 (내진설계범주 C)""")

response1 = chat(messages, system=structural_system)
add_assistant_message(messages, response1)
print("=== 1턴: 초기 설계 검토 ===")
print(response1)

# 2턴: 내진등급 변경에 따른 재검토
add_user_message(messages, """내진등급이 '특등급 (내진설계범주 D)'으로 변경되었습니다.
변경된 조건에서 기존 설계가 여전히 적합한지 재검토해주세요.""")

response2 = chat(messages, system=structural_system)
add_assistant_message(messages, response2)
print("\n=== 2턴: 내진등급 변경 재검토 ===")
print(response2)

# 3턴: 개선안 요청
add_user_message(messages, "부적합 항목에 대한 개선안을 제시해주세요.")

response3 = chat(messages, system=structural_system)
add_assistant_message(messages, response3)
print("\n=== 3턴: 개선안 ===")
print(response3)

=== 1턴: 초기 설계 검토 ===
RC 기둥 설계 적정성을 KDS 기준에 따라 검토하겠습니다.

## 1. 기본 정보 정리
- 기둥 단면: 500mm × 500mm
- fck = 24 MPa, fy = 400 MPa (SD400)
- 주근: 8-D25 (As = 8 × 490.9 = 3,927 mm²)
- Pu = 2,500 kN, Mu = 150 kN·m
- 내진설계범주 C

## 2. 적용 기준 및 검토 항목

### 2.1 최소 단면 치수 검토
**적용기준**: KDS 14 20 22 4.3.1
- 최소 단면 치수: 300mm 이상
- **검토결과**: 500mm > 300mm ✓ **적합**

### 2.2 철근비 검토
**적용기준**: KDS 14 20 22 4.3.2

**계산과정**:
- 총 단면적 (Ag) = 500 × 500 = 250,000 mm²
- 철근비 (ρ) = As/Ag = 3,927/250,000 = 0.0157 = 1.57%

**기준**:
- 최소 철근비: 0.8% ≤ ρ
- 최대 철근비: ρ ≤ 8% (일반), ρ ≤ 6% (내진)

**검토결과**: 0.8% < 1.57% < 6% ✓ **적합**

### 2.3 내진 상세 검토
**적용기준**: KDS 41 17 00 9.4

#### 2.3.1 주근 배치 (KDS 41 17 00 9.4.1.2)
- 최소 주근 개수: 4개 이상
- 각 모서리에 1개 이상 배치
- **검토결과**: 8-D25 배치 ✓ **적합**

#### 2.3.2 주근 직경 및 간격 (KDS 41 17 00 9.4.1.3)
- 최소 주근 직경: D13 이상
- 주근 간격: 300mm 이하
- **계산**: 주근 간격 ≈ (500-80)/3 = 140mm < 300mm
- **검토결과**: D25 > D13, 간격 140mm < 300mm ✓ **적합**

### 2.4 축력 검토
**적용기준**: KDS 14 20 22 4.3.3

**순축력비 계산**:
- Ag = 250,000 mm²
- 순축력비

In [15]:
# 구조 계산 — 정확성 우선 (temperature = 0.0)
messages_calc = []
add_user_message(messages_calc,
    "500x500 RC 기둥, fc'=24MPa, fy=400MPa일 때 "
    "축 하중 강도 Pn을 KDS 기준으로 계산하라."
)
result_precise = chat(messages_calc, temperature=0.0)
print("=== Temperature 0.0 (구조 계산) ===")
print(result_precise)

# 설계 아이디어 — 창의성 우선 (temperature = 0.9)
messages_idea = []
add_user_message(messages_idea,
    "20층 주거 건물의 횡력저항시스템으로 "
    "가능한 구조 대안을 브레인스토밍하라."
)
result_creative = chat(messages_idea, temperature=0.9)
print("\n=== Temperature 0.9 (브레인스토밍) ===")
print(result_creative)

=== Temperature 0.0 (구조 계산) ===
500×500 RC 기둥의 축 하중 강도를 KDS 기준으로 계산하겠습니다.

## 주어진 조건
- 기둥 단면: 500mm × 500mm
- fc' = 24 MPa
- fy = 400 MPa

## KDS 14 20 22 기준 축 하중 강도 계산

### 1. 기본 공식
KDS에 따른 축 하중 강도:
**Pn = 0.85 × fc' × (Ag - Ast) + fy × Ast**

여기서:
- Ag = 총 단면적
- Ast = 철근 단면적

### 2. 단면적 계산
**Ag = 500 × 500 = 250,000 mm² = 0.25 m²**

### 3. 철근비 가정
KDS 기준:
- 최소 철근비: 0.01
- 최대 철근비: 0.08

일반적인 설계에서 철근비 1% ~ 3% 사용

**철근비 2% 가정 시:**
Ast = 0.02 × 250,000 = 5,000 mm²

### 4. 축 하중 강도 계산
Pn = 0.85 × 24 × (250,000 - 5,000) + 400 × 5,000
   = 0.85 × 24 × 245,000 + 400 × 5,000
   = 4,998,000 + 2,000,000
   = **6,998,000 N = 6,998 kN**

### 5. 철근비별 축 하중 강도

| 철근비 | Ast (mm²) | Pn (kN) |
|--------|-----------|---------|
| 1.0%   | 2,500     | 6,074   |
| 1.5%   | 3,750     | 6,536   |
| 2.0%   | 5,000     | 6,998   |
| 2.5%   | 6,250     | 7,460   |
| 3.0%   | 7,500     | 7,922   |

## 결론
500×500 RC 기둥의 축 하중 강도는 철근비에 따라 **약 6,000 ~ 8,000 kN** 범위이며, 일반적인 철근비 2% 적용 시 **약 7,000 kN**입니다.

**참고:** 실제 설계